In [ ]:
%load_ext autoreload
%autoreload 2
import jax.numpy as jnp
import jax_sn
import jax_sbgeom as jsb
import jax
import matplotlib.pyplot as plt
import numpy as onp
import equinox as eqx
jax.config.update("jax_enable_x64", True)
from jax_sbgeom.flux_surfaces import FluxSurfaceNormalExtendedNoPhi, ToroidalExtent, FluxSurface
from jax_sn.tools.memory_analysis import _filter_jit_f_to_memory
vmec_file = ["/home/tbogaarts/stellarator_paper/base_data/vmecs/helias5_vmec.nc4"][0]
fs_base = FluxSurfaceNormalExtendedNoPhi.from_hdf5(vmec_file)

import pyvista as pv
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh

In [ ]:
n_plasma = 10
n_sol = 5
s_mesh      = jnp.concatenate([jnp.linspace(0.0, 1.0, n_plasma)**2, jnp.linspace(1.0, 1.2, n_sol+1)[1:]])
tor_extent  = ToroidalExtent.half_module(fs_base)
n_theta     = 51
n_phi       = 62
source_kwargs = {
    "n0"    : 2.4e20,
    "Ti0"   : 15.2,
    "nalpha": 0.35,
    "Tialpha": 1.2,
}


In [ ]:
from wall_opt_functions import interpolate_theta_phi_d, SOLSettings, setup_plasma_mgop, ParametrisedDistance, d_grid_to_domain


In [ ]:
from jax_sn.operators.multi_group_operator import MultiGroupOperatorScan
from jax_sn.parametrisation.multigroup_parametrisation import MultigroupOperatorParametrisedVerticesXS, VerticesXSTheta
from abc import abstractmethod

fs_base = FluxSurfaceNormalExtendedNoPhi.from_hdf5(vmec_file)

In [ ]:
d_base = 0.2

MGOP, source_eg0_all_angles, sol_settings = setup_plasma_mgop(fs_base, tor_extent, n_plasma, n_sol, d_base, n_theta, n_phi, source_kwargs, degree=1, batch_size=100)


In [ ]:
from wall_opt_functions import interpolate_theta_phi_d, recreate_mesh, create_sol_theta, plot_resulting_d, d_grid_to_values
class PhiInterpThetaFourier(ParametrisedDistance):    
    parameters : jnp.ndarray  # [n_phi, 1 + 2*K]
    d_min  : float = 0.1

    def interpolate_d(self, theta, phi, toroidal_extent):
        return self.d_min + 0.2 * jnp.exp(interpolate_theta_phi_d(self. parameters, theta,phi , toroidal_extent))
    
    def penalise_roughness(self):
        ak = self.parameters[:, 1::2]   # shape (n_phi, K)
        bk = self.parameters[:, 2::2]
        p = self.parameters
        n_phi, n_coeff = p.shape
        K = (n_coeff - 1) // 2

        a0 = p[:, 0]
        ak = p[:, 1::2]
        bk = p[:, 2::2]

        # θ-roughness (spectral)
        k = jnp.arange(1, K + 1)
        theta_energy = jnp.sum((k**2)[None, :] * (ak**2 + bk**2))

        # φ-roughness (finite difference)
        dphi_a0 = a0[1:] - a0[:-1]
        dphi_ak = ak[1:] - ak[:-1]
        dphi_bk = bk[1:] - bk[:-1]

        phi_energy = jnp.sum(dphi_a0**2) \
                + jnp.sum(dphi_ak**2) \
                + jnp.sum(dphi_bk**2)

        return theta_energy + phi_energy


In [ ]:
n_phi_params = 10
K_fourier = 4

d0 = jnp.zeros((n_phi_params, 1 + 2 * K_fourier))


In [ ]:
d_init = PhiInterpThetaFourier(d0)

In [ ]:
d_init.interpolate_d(jnp.array(0.1),jnp.array( 0.2), tor_extent)

In [ ]:
from wall_opt_functions import d_grid_to_values
def loss_block_fluxes(block_fluxes : jnp.ndarray, d_grid, l_smooth = 1e-3):    
    flux_mean = jnp.mean(block_fluxes)
    return jnp.mean(jax.nn.softplus((block_fluxes -  flux_mean) / flux_mean )**2) + d_grid.penalise_roughness() * l_smooth



@eqx.filter_jit
def d_grid_to_loss(d_grid, MGOP : MultiGroupOperatorScan, source_eg0_all_angles : jnp.ndarray, sol_settings : jnp.ndarray) -> jnp.ndarray:
    jax.debug.print("Computing loss for d_grid with shape: {d_grid}", d_grid=d_grid)
    return loss_block_fluxes(d_grid_to_values(d_grid, MGOP, source_eg0_all_angles, sol_settings), d_grid)
dloss_dgrid = eqx.filter_jit(jax.value_and_grad(d_grid_to_loss, argnums = 0))

In [ ]:
import optax
d0 = d_init

lbfgs = optax.adam(3e-2)
#value_and_grad_function =  d_grid_to_loss

opt_state = lbfgs.init(d0)
params    = d0

def loss_fn(params : jnp.ndarray):
    return d_grid_to_loss(params, MGOP, source_eg0_all_angles, SOLSettings(fs_base, tor_extent, n_theta, n_phi, n_sol))



In [ ]:
total_params = []
for i in range(60):
    print(i)
    value, grad = dloss_dgrid(params, MGOP, source_eg0_all_angles, SOLSettings(fs_base, tor_extent, n_theta, n_phi, n_sol))
    updates, opt_state = lbfgs.update(
            grad, opt_state, params=params, value=value, grad=grad, value_fn=loss_fn
        )
    params = PhiInterpThetaFourier(params.parameters+ updates.parameters)# params = optax.apply_updates(params, updates)
    print(f"Step {i}, loss = {value}")
    print(params)
    total_params.append(params)
    plot_resulting_d(params, tor_extent)

    # print(params)

In [ ]:
values_one_less = d_grid_to_values(total_params[-1], MGOP, source_eg0_all_angles, SOLSettings(fs_base, tor_extent, n_theta, n_phi, n_sol))
values_zero = d_grid_to_values(d_init, MGOP, source_eg0_all_angles, SOLSettings(fs_base, tor_extent, n_theta, n_phi, n_sol))

In [ ]:
plt.pcolormesh(values_one_less.T)
plt.colorbar()

plt.figure()
plt.pcolormesh(values_zero.T)
plt.colorbar()

plt.figure()
plt.pcolormesh(values_one_less.T/values_zero.T)
plt.colorbar()

In [ ]:
from wall_opt_functions import d_grid_to_domain, d_grid_to_flux_elements
import pyvista as pv

vertices_new = d_grid_to_domain(total_params[-1], MGOP, SOLSettings(fs_base, tor_extent, n_theta, n_phi, n_sol))
vertices_start = d_grid_to_domain(d_init, MGOP, SOLSettings(fs_base, tor_extent, n_theta, n_phi, n_sol))

connectivity = MGOP.transport_operator.transport_operator.solution_domain_data.domain.connectivity

In [ ]:
block_flux = d_grid_to_flux_elements(total_params[-1], MGOP, source_eg0_all_angles, SOLSettings(fs_base, tor_extent, n_theta, n_phi, n_sol))

In [ ]:
plotter = pv.Plotter()


plotter.add_mesh(mesh_to_pyvista_mesh(vertices_new, connectivity), scalars = block_flux, opacity=1.0, label="Optimized", clim = [jnp.min(block_flux), jnp.max(values_one_less)])
plotter.show()

In [ ]:
import meshio 

In [ ]:
mesh_plasma = meshio.Mesh(onp.array(vertices_new), [("tetra", onp.array(connectivity)[:-n_theta * (n_phi - 1) * 6 * n_sol])])
mesh_plasma.write("/mnt/d/vtks/animation_optimization_fw/plasma.vtk")

In [ ]:
for i, d_grid in enumerate(total_params):
    block_flux=d_grid_to_flux_elements(d_grid, MGOP, source_eg0_all_angles, SOLSettings(fs_base, tor_extent, n_theta, n_phi, n_sol));
    vert_new = d_grid_to_domain(d_grid, MGOP, SOLSettings(fs_base, tor_extent, n_theta, n_phi, n_sol)); \
    mesh = meshio.Mesh(onp.array(vert_new), [("tetra", onp.array(connectivity))], cell_data={"block_flux": [block_flux]})
    mesh_ll = meshio.Mesh(onp.array(vert_new), [("tetra", onp.array(connectivity)[-n_theta * (n_phi-1) * 6:,:])], cell_data={"block_flux": [block_flux[-n_theta *( n_phi -1) * 6:]]})
    print(mesh)
    print(mesh_ll)
    mesh.write("/mnt/d/vtks/animation_optimization_fw/whole_domain_"+str(i)+".vtk")
    mesh_ll.write("/mnt/d/vtks/animation_optimization_fw/last_layer_"+str(i)+".vtk")



In [ ]:
plotter = pv.Plotter()

plotter.add_mesh(mesh_to_pyvista_mesh(vertices_start, connectivity), color="red", opacity=0.5, label="Start")    
plotter.add_mesh(mesh_to_pyvista_mesh(vertices_new, connectivity), color="blue", opacity=1.0, label="Optimized")
plotter.show()
